<a href="https://colab.research.google.com/github/57sg77g8sk-ops/ADALL_github/blob/main/ADALL_Exam_Cheatsheet_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ADALL Exam Cheatsheet — Data Preparation, Modelling, Evaluation & Justification

**Colab-ready | Based only on Weeks 1–6 study materials**

Use this notebook in two ways:

1. **Last-minute revision:** read the red-flag rules, metric tables, and answer templates.
2. **Practical test:** copy a relevant code block, change the dataset/target names, run it, inspect the output, and explain your judgement.

> **Core exam mindset:** Keep it simple. Use a correct workflow, check your outputs, prevent leakage, compare models fairly, and justify decisions with evidence.

### Contents

1. 15-minute emergency review
2. Business framing: modelling objective, JTBD, RACI, deployment lenses
3. End-to-end analytics workflow
4. Dataset inspection and payload text
5. Cleaning, leakage, and feature engineering
6. Regression vs classification
7. Split, cross-validation, preprocessing, and pipelines
8. Regression metrics and model template
9. Regression diagnosis and final judgement
10. Classification metrics and model template
11. FI, permutation importance, and SHAP
12. Safe LLM use and prompt templates
13. Colab/GitHub/versioning reminders
14. Common mistakes and rapid-fire recall

## 1. If the exam starts soon: memorise these rules

| Rule | What to remember |
|---|---|
| Define the target | Say exactly what is predicted and why it is useful. |
| Decide task type | Numeric value/count → usually regression. Category/label → classification. |
| Prevent leakage | A predictor must not contain the answer or information unavailable at prediction time. |
| Split before learning | Fit preprocessing and models on training data; keep the test set for the final check. |
| Use a pipeline | It applies the same preprocessing during training, validation, testing, and deployment. |
| Compare fairly | Same data split, preprocessing, CV method, and metric for every model. |
| Use CV for selection | Select using training-data CV; use the test set once for final evaluation. |
| Beat a dummy baseline | A sophisticated model must outperform a simple guess. |
| Read more than one metric | Scores show different weaknesses; plots and error tables reveal where errors occur. |
| Explain cautiously | FI, pFI, and SHAP explain model behaviour—not real-world causation. |
| Treat warnings as warnings | High cardinality, correlation, and outliers require inspection, not automatic deletion. |
| LLM = support tool | Check its assumptions, code, privacy impact, and evidence. You own the decision. |

### High-value sentences

- **Leakage:** “I removed ___ because it directly defines/reveals the target, so keeping it would make evaluation unrealistically optimistic.”
- **MAE:** “The model is wrong by about ___ target units on average.”
- **CV:** “Cross-validation reduces dependence on one lucky train/validation split.”
- **Pipeline:** “The pipeline fits preprocessing on training data and applies it consistently to unseen data.”
- **Explainability:** “This shows what the model relies on; it does not prove that the feature causes the outcome.”

## 2. Business framing before code

### Business problem → modelling objective

A business problem is written in business language. A modelling objective is measurable and technical.

**Template**

> Build a **[regression/classification]** model to predict **[target]** using **[available predictors]** for **[user]**, so they can **[decision/value]**. Evaluate it using **[metric]** because **[business meaning]**.

**Example**

> Build a regression model to predict `Price_SGD` using laptop specifications for the pricing team, so prices can be set more consistently. Use MAE because it is the average error in Singapore dollars.

### JTBD (Job-To-Be-Done)

Focus on the user’s desired outcome:

> When **[situation]**, the user wants to **[job]**, so that **[outcome]**.

Example outcome: estimate next-quarter sales or estimate a fair laptop price.

### RACI

| Role | Meaning | Key exam point |
|---|---|---|
| **R — Responsible** | Does the work | Can be more than one; keep manageable. |
| **A — Accountable** | Owns and approves the outcome | Usually exactly one per task. |
| **C — Consulted** | Gives two-way expert input | Too many can slow progress. |
| **I — Informed** | Receives one-way updates | Does not perform or approve the task. |

### Six deployment lenses

1. **Business fit:** who uses the output, what decision changes, what value is created?
2. **Interpretability:** who needs an explanation, and how detailed must it be?
3. **Data quality:** missingness, ownership, definition, freshness, invalid values.
4. **Risk of wrong predictions:** financial, customer/staff, trust, and reputation impact.
5. **Timeline:** are predictors available when the prediction is needed?
6. **Maintenance:** who monitors performance, detects drift, and triggers updates?

> A good test score is necessary, but deployment also depends on risk, timing, data quality, maintenance, and business fit.

## 3. End-to-end workflow

```text
Business problem
→ modelling objective and target
→ load and inspect data
→ identify data risks and leakage
→ clean with evidence
→ define X and y
→ hold out test data
→ build preprocessing pipeline
→ train simple baseline
→ compare models with CV
→ final test evaluation
→ error diagnosis and explainability
→ justified recommendation
→ monitor after deployment
```

### What `X` and `y` mean

- `X`: predictor/input columns.
- `y`: target/output to predict.
- Never leave the target—or a direct answer source—inside `X`.

## 4. Setup and quick dataset inspection

Run only the imports you need. If the exam provides the loading cell, run it before rewriting anything.

In [ ]:
# Core imports for most ADALL regression/classification tasks
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, ShuffleSplit, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42

# Example loading patterns — choose ONE and edit it
# df = pd.read_csv("https://raw.githubusercontent.com/USER/REPO/main/file.csv")
# df = pd.read_csv("/content/file.csv")

# Quick checks after df exists:
# print("Shape:", df.shape)
# display(df.head())
# df.info()
# print(df.columns.tolist())
# print("\nMissing values:\n", df.isna().sum().sort_values(ascending=False).head(15))
# print("\nDuplicate rows:", df.duplicated().sum())

### Minimum inspection questions

1. What does one row represent?
2. What is the target, and is it numeric or categorical?
3. Which columns are numeric and categorical?
4. Are values missing, duplicated, invalid, constant, or suspicious?
5. Are any columns IDs, post-outcome information, or direct components of the target?
6. Will every predictor be available at prediction time?
7. Do ranges, categories, outliers, and target distribution make business sense?

## 5. Payload text: summarise before asking an LLM

**Definition:** payload text is a compact dataset profile prepared locally and sent to an LLM as context.

| Method | Advantage | Limitation |
|---|---|---|
| First 10 rows | Easy to read | May not represent the whole dataset |
| Full dataset | More row detail | Privacy, cost, size, and governance risks |
| Payload text | Compact whole-dataset overview | Can hide row-level patterns and rare combinations |

Creating a string does **not** send data. The data is sent only when an API/chat request is made.

In [ ]:
def build_payload_text(df):
    # Create a compact dataset profile locally. This function sends nothing.
    parts = []
    parts.append(f"=== SHAPE ===\nRows: {df.shape[0]}\nColumns: {df.shape[1]}")
    parts.append("=== DTYPES ===\n" + df.dtypes.to_string())
    parts.append("=== MISSING VALUES ===\n" + pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2)
    }).to_string())
    parts.append("=== UNIQUE COUNTS ===\n" +
                 df.nunique(dropna=False).to_frame("unique_count").to_string())
    parts.append("=== NUMERIC SUMMARY ===\n" +
                 df.describe(include="number").round(2).to_string())

    cat_cols = df.select_dtypes(include=["object", "category"]).columns
    cat_lines = []
    for col in cat_cols:
        cat_lines.append(f"\n[{col}]\n{df[col].value_counts(dropna=False).head(10)}")
    parts.append("=== TOP CATEGORICAL VALUES ===\n" +
                 ("".join(cat_lines) if cat_lines else "No categorical columns"))

    parts.append("=== NUMERIC CORRELATIONS ===\n" +
                 df.corr(numeric_only=True).round(2).to_string())
    possible_ids = [c for c in df.columns if df[c].nunique(dropna=False) == len(df)]
    constants = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
    parts.append(
        "=== WARNINGS TO INSPECT ===\n"
        f"Possible ID-like/unique columns: {possible_ids}\n"
        f"Constant columns: {constants}\n"
        f"Duplicate rows: {df.duplicated().sum()}"
    )
    return "\n\n".join(parts)

# After df exists:
# payload_text = build_payload_text(df)
# print(payload_text[:12000])

## 6. Cleaning, leakage, and feature engineering

### Warnings are not automatic actions

| Warning | Possible concern | Important exception |
|---|---|---|
| Unique values | ID/reference may encourage memorisation | A continuous target can naturally be unique |
| High cardinality | Too many sparse categories | Product/model names may contain useful signal |
| Outlier | Error or unusual case | A valid premium product may be extreme |
| High correlation | Redundant predictors | Both may retain distinct business meaning |
| Constant column | No predictive variation | May still document dataset scope, but not help prediction |

### Leakage test

Ask these questions for every predictor:

1. Does it directly define or calculate the target?
2. Was it created after the outcome happened?
3. Would it be known at the exact time the prediction is requested?
4. Is it a disguised copy/proxy of the answer?

**Study-material example:** if `num_failed_subjects` is calculated from `G1`, `G2`, and `G3`, those grade columns must be removed from `X`.

```python
df["num_failed_subjects"] = (df[["G1", "G2", "G3"]] < 10).sum(axis=1)
X = df.drop(columns=["num_failed_subjects", "G1", "G2", "G3"])
y = df["num_failed_subjects"]
```

### Safe cleaning order

1. Copy the original: `cleaned_df = df.copy()`.
2. Verify ranges and business rules.
3. Handle missing values deliberately.
4. Remove confirmed index/ID, constant, duplicate, or leakage columns.
5. Standardise category text only when justified.
6. Check shape, columns, missingness, and target again.

> Do not clip outliers, group categories, or drop high-cardinality columns without checking business meaning and validation results.

### Feature engineering

A feature is a hypothesis, not an improvement until validation proves it.

- Build only from columns available at prediction time.
- Do not use the target.
- Change one idea at a time when revising.
- Compare CV performance before and after.

## 7. Regression or classification?

| Target/question | Usual task |
|---|---|
| Laptop price, sales, demand, continuous measurement | Regression |
| Numeric count such as number of failures | Often regression; can be a count-model edge case |
| Malignant vs benign, pass vs fail, named category | Classification |
| “Failed at least one subject?” (yes/no) | Binary classification |

The task depends on how the target is defined—not merely how the source data looks.

## 8. Train/test split, CV, and pipelines

### Correct roles

- **Training data:** fits preprocessing and model parameters.
- **Validation/CV data:** compares choices within the training data.
- **Test data:** final check after model selection.
- **Deployment data:** future data; may drift away from the training distribution.

### Stratification

- Classification: commonly use `stratify=y`.
- Ordinary regression: usually do not stratify.
- Small integer-count regression is an edge case; stratification may help preserve proportions but can fail if values are rare.

### Why pipelines matter

A pipeline prevents inconsistent preprocessing and reduces leakage because transformations are fitted within each training fold, then applied to validation/test data.

### Fair comparison checklist

Use the same:

- training/test split,
- preprocessing,
- CV splitter,
- scoring metric,
- random state where applicable.

## 9. Regression metrics

| Metric | Meaning | Better |
|---|---|---|
| **MAE** | Mean absolute error; average error in target units | Lower |
| **RMSE** | Like MAE but punishes large errors more | Lower |
| **R²** | Fit relative to predicting the mean | Higher |
| **MAPE** | Average percentage error | Lower; unstable when actual values are zero/near zero |

Formulas:

- `MAE = mean(|prediction − actual|)`
- `RMSE = sqrt(mean((prediction − actual)²))`
- `R² = 1 − model squared error / mean-baseline squared error`

Interpretation reminders:

- MAE = 100 does **not** mean every prediction is exactly 100 wrong.
- RMSE much larger than MAE suggests some large errors.
- R² can be negative when the model is worse than predicting the mean.
- Always compare with `DummyRegressor(strategy="mean")`.

## 10. Regression master template

Change only the target/leakage columns and model settings required by the question. This template compares models by CV on the training data, then evaluates the selected model on the untouched test set.

In [ ]:
# ===== REGRESSION TEMPLATE =====
# Assumes a DataFrame named df already exists.

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Optional in Colab:
# !pip -q install xgboost
from xgboost import XGBRegressor

TARGET = "CHANGE_ME"
LEAKAGE_COLS = []  # e.g. ["G1", "G2", "G3"]

# 1) Define X and y
X = df.drop(columns=[TARGET] + LEAKAGE_COLS)
y = df[TARGET]

# 2) Final holdout split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

# 3) Column types and shared preprocessing
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", "passthrough", num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

# 4) Candidate models
models = {
    "Decision Tree": DecisionTreeRegressor(
        max_depth=5, random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "XGBoost": XGBRegressor(
        n_estimators=150, max_depth=4, learning_rate=0.08,
        subsample=0.9, colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=RANDOM_STATE, n_jobs=1
    )
}

# 5) Select using CV inside training data
cv = ShuffleSplit(n_splits=5, test_size=0.2, random_state=RANDOM_STATE)
rows, fitted = [], {}

for name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    cv_out = cross_validate(
        pipe, X_train, y_train,
        cv=cv,
        scoring="neg_mean_absolute_error",
        return_train_score=True,
        n_jobs=-1
    )
    train_mae = -cv_out["train_score"]
    val_mae = -cv_out["test_score"]
    rows.append({
        "model": name,
        "cv_train_mae": train_mae.mean(),
        "cv_val_mae": val_mae.mean(),
        "cv_val_std": val_mae.std(),
        "train_val_gap": val_mae.mean() - train_mae.mean()
    })
    fitted[name] = pipe

cv_results = pd.DataFrame(rows).sort_values("cv_val_mae")
display(cv_results.round(3))

# 6) Fit selected model once and evaluate untouched test set
best_name = cv_results.iloc[0]["model"]
best_model = fitted[best_name]
best_model.fit(X_train, y_train)
pred = best_model.predict(X_test)

test_mae = mean_absolute_error(y_test, pred)
test_rmse = np.sqrt(mean_squared_error(y_test, pred))
test_r2 = r2_score(y_test, pred)

# 7) Dummy baseline
dummy = DummyRegressor(strategy="mean")
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)
dummy_mae = mean_absolute_error(y_test, dummy_pred)

summary = pd.DataFrame({
    "model": ["Mean dummy baseline", best_name],
    "test_mae": [dummy_mae, test_mae],
    "test_rmse": [
        np.sqrt(mean_squared_error(y_test, dummy_pred)),
        test_rmse
    ],
    "test_r2": [r2_score(y_test, dummy_pred), test_r2]
})
display(summary.round(3))

print("Selected model:", best_name)
print("MAE improvement vs dummy:", round(dummy_mae - test_mae, 3))

### How to choose from the regression table

1. Lower CV validation MAE is better.
2. Smaller CV standard deviation means more stable performance.
3. A large train–validation MAE gap suggests overfitting.
4. If two models are very close, prefer the simpler/easier-to-maintain model.
5. The final test result should broadly support the CV decision.
6. The chosen model should meaningfully beat the dummy baseline.

## 11. Regression error diagnosis

One average score can hide systematic problems. Inspect:

- largest individual errors,
- over- vs under-prediction,
- actual vs predicted plot,
- residual/error pattern,
- performance by meaningful bands/groups,
- group sample sizes.

In [ ]:
# Run after the regression template
error_df = X_test.copy()
error_df["actual"] = y_test.to_numpy()
error_df["predicted"] = pred
error_df["error"] = error_df["predicted"] - error_df["actual"]
error_df["absolute_error"] = error_df["error"].abs()
error_df["error_direction"] = np.select(
    [error_df["error"] > 0, error_df["error"] < 0],
    ["over-predicted", "under-predicted"],
    default="exact"
)

display(error_df.sort_values("absolute_error", ascending=False).head(10))
display(error_df["error_direction"].value_counts())

# Actual vs predicted: good points sit near the diagonal
plt.figure(figsize=(6, 6))
plt.scatter(error_df["actual"], error_df["predicted"], alpha=0.7)
lo = min(error_df["actual"].min(), error_df["predicted"].min())
hi = max(error_df["actual"].max(), error_df["predicted"].max())
plt.plot([lo, hi], [lo, hi], color="black")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Actual vs Predicted")
plt.show()

# Residual/error plot: look for pattern around zero
plt.figure(figsize=(8, 4))
plt.scatter(error_df["actual"], error_df["error"], alpha=0.7)
plt.axhline(0, color="black")
plt.xlabel("Actual")
plt.ylabel("Prediction error (predicted - actual)")
plt.title("Error by Actual Value")
plt.show()

# Optional quartile bands (MAPE excluded because zero actuals can break it)
error_df["actual_band"] = pd.qcut(error_df["actual"], q=4, duplicates="drop")
band_error = error_df.groupby("actual_band", observed=True).agg(
    count=("absolute_error", "size"),
    mean_actual=("actual", "mean"),
    mae=("absolute_error", "mean"),
    median_absolute_error=("absolute_error", "median")
)
display(band_error.round(3))

### Regression judgement template

> The selected model is **___**, chosen because its CV validation MAE was **___** with variability of **___**. Compared with the mean baseline, it reduced test MAE from **___** to **___**, which is **___%** improvement. Its test MAE means predictions are about **___ target units** away from actual values on average. The plots/error table show **___**. Performance is weakest for **___**, although this group has **___** records, so **___**. I would **recommend / not recommend** it for **___** with **___ safeguard**. A sensible next step is **___**.

## 12. Classification metrics

### Confusion matrix terms

| Term | Meaning |
|---|---|
| TP | Correctly predicted positive |
| TN | Correctly predicted negative |
| FP | Predicted positive but actually negative |
| FN | Predicted negative but actually positive |

| Metric | Formula/idea | Use |
|---|---|---|
| Accuracy | `(TP + TN) / all` | Overall correctness; can mislead with imbalance |
| Precision | `TP / (TP + FP)` | When false positives are costly |
| Recall | `TP / (TP + FN)` | When missing positives is costly |
| F1 | Harmonic mean of precision and recall | Balance precision and recall |
| MCC | Balanced correlation-like score from −1 to 1 | Strong summary under class imbalance |

Always confirm which class is treated as the positive class. Use more than one metric and inspect the confusion matrix.

## 13. Classification master template

In [ ]:
# ===== CLASSIFICATION TEMPLATE =====
# Assumes DataFrame df exists.

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, confusion_matrix, classification_report
)
from xgboost import XGBClassifier

TARGET = "CHANGE_ME"
LEAKAGE_COLS = []

X = df.drop(columns=[TARGET] + LEAKAGE_COLS)
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", "passthrough", num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=4, random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=150, max_depth=3, learning_rate=0.08,
        subsample=0.9, colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=RANDOM_STATE, n_jobs=1
    )
}

rows, fitted = [], {}
for name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    pipe.fit(X_train, y_train)
    pred_cls = pipe.predict(X_test)
    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred_cls),
        "precision": precision_score(y_test, pred_cls, zero_division=0),
        "recall": recall_score(y_test, pred_cls, zero_division=0),
        "f1": f1_score(y_test, pred_cls, zero_division=0),
        "mcc": matthews_corrcoef(y_test, pred_cls)
    })
    fitted[name] = pipe

classification_results = pd.DataFrame(rows).sort_values("mcc", ascending=False)
display(classification_results.round(3))

best_cls_name = classification_results.iloc[0]["model"]
best_cls_model = fitted[best_cls_name]
best_cls_pred = best_cls_model.predict(X_test)

print("Selected:", best_cls_name)
print("\nConfusion matrix:\n", confusion_matrix(y_test, best_cls_pred))
print("\nClassification report:\n", classification_report(y_test, best_cls_pred))

### Classification judgement template

> I selected **___** based on **___**, while also checking **___** and the confusion matrix. The model correctly/incorrectly classified **___**. The most important practical error is **FP/FN** because **___**. I would **recommend / not recommend** it for **___**, with **human review / threshold review / monitoring / other safeguard** because **___**.

## 14. Explainability: FI, pFI, and SHAP

| Method | Main question | Main caveat |
|---|---|---|
| Tree Feature Importance (FI) | Which features did the tree model use strongly for splits? | Internal/model-specific; can favour certain features |
| Permutation Feature Importance (pFI) | How much does the chosen score worsen when a feature is shuffled? | Depends on dataset, metric, repeats, and correlated features |
| Global SHAP | Which features have the largest average effect on model output? | Explains model behaviour, not causation |
| Local SHAP | Which features pushed one prediction higher/lower? | One row does not describe the whole model |

The rankings can disagree without either method being automatically “wrong.”

In [ ]:
# Permutation importance works with a fitted pipeline and original columns.
from sklearn.inspection import permutation_importance

# Regression example:
# pfi = permutation_importance(
#     best_model, X_test, y_test,
#     n_repeats=10, random_state=RANDOM_STATE,
#     scoring="neg_mean_absolute_error"
# )

# Classification example:
# pfi = permutation_importance(
#     best_cls_model, X_test, y_test,
#     n_repeats=10, random_state=RANDOM_STATE,
#     scoring="f1"
# )

# pfi_df = pd.DataFrame({
#     "feature": X_test.columns,
#     "importance_mean": pfi.importances_mean,
#     "importance_std": pfi.importances_std
# }).sort_values("importance_mean", ascending=False)
# display(pfi_df.head(10))

### SHAP reading guide

- **Bar plot:** ranks global mean absolute SHAP impact.
- **Beeswarm:** shows global importance plus direction/distribution.
- **Dependence/scatter plot:** relates one feature’s value to SHAP impact.
- **Waterfall:** explains one row from baseline output to final prediction.

**Careful wording**

> “The model seems to rely strongly on ___.”  
> “For this row, ___ pushed the model output higher/lower.”  
> “This is model evidence, not proof that changing ___ will cause the real-world outcome to change.”

In [ ]:
# SHAP skeleton for a fitted tree model with numeric features
# !pip -q install shap xgboost
# import shap
#
# X_explain = X_test.iloc[:60].copy()
# explainer = shap.TreeExplainer(xgb_model)
# shap_values = explainer(X_explain)
#
# shap.plots.bar(shap_values, max_display=12)       # global ranking
# shap.plots.beeswarm(shap_values, max_display=12) # global direction
# shap.plots.waterfall(shap_values[0], max_display=12)  # one row
#
# local_table = pd.DataFrame({
#     "feature": X_explain.columns,
#     "value": X_explain.iloc[0].values,
#     "shap_value": shap_values[0].values
# })
# local_table["absolute_shap"] = local_table["shap_value"].abs()
# display(local_table.sort_values("absolute_shap", ascending=False).head(10))

## 15. Safe and useful LLM assistance

### Before accepting LLM-generated code

Check whether it:

1. uses the correct task type and target;
2. removes leakage;
3. handles numeric and categorical columns;
4. keeps preprocessing inside a pipeline;
5. uses CV for comparison and preserves the test set;
6. uses the requested metric;
7. invents columns, assumptions, or causes;
8. performs unnecessary tuning that wastes exam time;
9. sends private/full data unnecessarily;
10. exposes API keys or tokens.

### Prompt formula

Give the LLM:

```text
Role + task + available evidence + constraints + required output format
```

Better prompt example:

```text
Generate beginner-friendly Python for a regression task using existing
X_train, X_test, y_train, and y_test.
Compare DecisionTreeRegressor, RandomForestRegressor, and XGBRegressor.
Use one shared ColumnTransformer, OneHotEncoder(handle_unknown="ignore"),
a Pipeline, 5-fold ShuffleSplit CV, and MAE.
Select using CV on training data, then evaluate once on the test set.
Do not use GridSearchCV and do not invent columns.
```

### Focused prompt: data-quality review

```text
You are helping a data analytics student review a dataset.

Target definition: [target and how it was created]
Dataset profile:
[payload_text]

Answer only:
1. data-quality risks to check,
2. possible leakage columns and reasons,
3. recommendations that require human judgement.

Do not invent facts. Do not say to drop a column only because it appears in a warning.
```

### Focused prompt: evidence-based model judgement

```text
Use only the evidence below. Do not invent causes.
Write a short judgement containing:
1. baseline comparison,
2. one strength,
3. one weakness,
4. one evidence limitation,
5. one next step.

[paste metrics, error summaries, group counts, and explainability results]
```

## 16. Colab, GitHub, AI coding tools, and versioning

### Colab + GitHub essentials

- Use the **raw GitHub URL** for `pd.read_csv`, not the normal preview page.
- Colab opened from GitHub does not automatically save every edit back to GitHub.
- Use **Save a copy in GitHub / Save in GitHub**, then commit intentionally.
- Keep train/test files and README files clearly named.
- Never upload API keys, passwords, or Colab secrets.
- Store secrets in Colab Secrets (`userdata.get(...)`), not in notebook cells.

### Versioning matters

Commit history:

- preserves earlier working versions,
- shows what changed,
- supports rollback and audit,
- helps explain model improvements.

### AI coding assistance levels from the study material

| Level | Typical assistance | Main concern |
|---|---|---|
| 0 | Static editor/tools | No logic understanding |
| 1 | Token/autocomplete | Mostly syntax-level help |
| 2 | Function/code-block generation | Limited wider context |
| 3 | Supervised multi-step coding/debugging | Human must verify logic |
| 4 | Agentic, multi-file support | Larger hallucination/logical-error blast radius |
| 5 | Autonomous coding concept | Reliability and production readiness |

The more autonomy a tool has, the more important human review, tests, permissions, versioning, and fail-safes become.

## 17. Common exam mistakes

| Mistake | Why it is wrong | Fix |
|---|---|---|
| Target left inside `X` | Direct leakage | Drop target before modelling |
| Target components kept as features | Model can reconstruct answer | Remove direct components |
| Preprocessing before split/CV | Validation information can leak | Put preprocessing in pipeline |
| Test set used repeatedly for model choice | Test is no longer independent | Select with CV; test once |
| Comparing models with different preprocessing/splits | Unfair comparison | Reuse the same pipeline/split/CV |
| Choosing “advanced” model by name | Complexity is not evidence | Use CV, stability, simplicity, test support |
| Reporting only accuracy | Can hide minority-class failure | Add F1/MCC/confusion matrix |
| Saying “SHAP proves cause” | Explanation is not causal inference | Say “the model relies on/prediction was pushed by” |
| Dropping every warning column | Warnings need context | Inspect, justify, validate |
| Blind LLM code execution | May assume wrong task/columns | Read, run, check, explain |
| Rounding regression predictions automatically | Changes output without instruction | Round only if required |
| Ignoring group size | Tiny groups produce unstable error estimates | Report counts and avoid overclaiming |

## 18. Rapid-fire recall

Try answering before opening the answers.

1. Why is a pipeline safer than manual preprocessing?
2. Why must `G1`, `G2`, and `G3` be removed after creating `num_failed_subjects`?
3. What is the difference between CV and the final test set?
4. Why does scikit-learn return negative MAE in scoring?
5. What does a positive prediction error (`predicted - actual`) mean?
6. Why compare against a dummy/mean baseline?
7. When can accuracy be misleading?
8. What is the difference between global and local SHAP?
9. Why can FI and pFI rankings differ?
10. Does creating `payload_text` send data to an LLM?

<details>
<summary><strong>Answers</strong></summary>

1. It fits and applies identical preprocessing consistently within training/CV/test workflows.
2. They directly calculate the target; keeping them leaks the answer.
3. CV compares choices within training data; the untouched test set is the final check.
4. Scikit-learn maximises scores, so it negates an error where lower is better.
5. The model over-predicted.
6. A useful ML model should beat a simple guess.
7. When classes are imbalanced or error costs differ.
8. Global SHAP summarises the model overall; local SHAP explains one prediction.
9. They ask different questions and react differently to correlation, model structure, test data, and metric.
10. No. Data is sent only when an API/chat request is made.
</details>

## 19. Final 5-minute answer checklist

Before submitting, confirm your written answer states:

- [ ] what one row represents;
- [ ] the target and task type;
- [ ] leakage columns removed and why;
- [ ] train/test/CV roles;
- [ ] preprocessing/pipeline used;
- [ ] model-selection evidence;
- [ ] final test metric interpreted in plain language;
- [ ] comparison with a simple baseline;
- [ ] one error pattern or limitation;
- [ ] a cautious recommendation and next step.

> Good answers are short, evidence-based, and careful. Do not explain every code line unless asked.

## Source study materials

This cheatsheet was consolidated from:

- W1S1 and W1S2: Introduction to Data Science and LLMs
- W2S1 and W2S2: Development Tools
- Week 3: Preparing Data and Building a Baseline Model
- Week 4: Modelling and Evaluation
- Week 5: Model Evaluation Practical Test Revision
- Week 6 Session 1: Classification, Feature Importance, and pFI
- Week 6 Session 2: SHAP Explanation Lab

No external material was added.